In [ ]:
"""
Hub Gene Identification Pipeline using py4cytoscape
====================================================
Inputs:
  - edges CSV: columns [source, target] (gene-gene interactions)
  - nodes CSV: columns [id, module] (gene name, module color)

Outputs per dataset:
  - Top 10 hub genes by Degree and MCC
  - Network visualization saved as PNG
  - Excel summary report across all datasets

Requirements:
  pip install py4cytoscape pandas openpyxl requests matplotlib

Cytoscape must be open and running before executing this script.
"""

import os
import time
import requests
import pandas as pd
import py4cytoscape as p4c
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# =============================================================================
# CONFIGURATION — edit these paths for your datasets
# =============================================================================

DATASETS = [
    {
        "name": "Dataset_1_Atherosclerosis",
        "cytoscape_edges_csv": "dataset1_edges.csv",   # columns: source, target
        "cytoscape_nodes_csv": "dataset1_nodes.csv",   # columns: id, module
    }
   
    # Add more datasets here following the same pattern
]

OUTPUT_DIR = "hub_gene_outputs"
CYTOHUBBA_METHODS = ["Degree", "MCC"]   # Methods to run in CytoHubba
TOP_N = 10                               # Number of hub genes to report
CHEA3_URL = "https://maayanlab.cloud/chea3/api/enrich/"  # ChEA3 API endpoint

# =============================================================================
# SETUP
# =============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

def check_cytoscape():
    """Verify Cytoscape is running."""
    try:
        version = p4c.cytoscape_version_info()
        print(f"Connected to Cytoscape {version['cytoscapeVersion']}")
        return True
    except Exception as e:
        print(f"ERROR: Cannot connect to Cytoscape. Make sure it is open.\n{e}")
        return False

def check_cytohubba():
    """Verify CytoHubba app is installed."""
    apps = p4c.get_installed_apps()
    installed = [a['appName'] for a in apps]
    if 'cytohubba' not in [a.lower() for a in installed]:
        print("WARNING: CytoHubba not found. Install it via Apps > App Manager in Cytoscape.")
        return False
    return True

# =============================================================================
# NETWORK LOADING
# =============================================================================

def load_network(dataset):
    """Load edge list and node attributes into Cytoscape."""
    name = dataset["name"]
    print(f"\n{'='*60}")
    print(f"Processing: {name}")
    print(f"{'='*60}")

    edges_df = pd.read_csv(dataset["edges_csv"])
    nodes_df = pd.read_csv(dataset["nodes_csv"])

    # Ensure correct column names
    edges_df.columns = edges_df.columns.str.strip()
    nodes_df.columns = nodes_df.columns.str.strip()

    # Validate
    assert "source" in edges_df.columns and "target" in edges_df.columns, \
        f"edges CSV must have 'source' and 'target' columns. Found: {list(edges_df.columns)}"
    assert "id" in nodes_df.columns and "module" in nodes_df.columns, \
        f"nodes CSV must have 'id' and 'module' columns. Found: {list(nodes_df.columns)}"

    print(f"  Edges: {len(edges_df)} | Nodes: {len(nodes_df)}")

    # Create network in Cytoscape
    network_suid = p4c.create_network_from_data_frames(
        nodes=nodes_df,
        edges=edges_df,
        source_id_list="source",
        target_id_list="target",
        node_id_list="id",
        title=name
    )

    # Import node attributes (module/color)
    p4c.load_table_data(
        data=nodes_df,
        data_key_column="id",
        table="node",
        table_key_column="name",
        network=network_suid
    )

    print(f"  Network created. SUID: {network_suid}")
    return network_suid, nodes_df, edges_df

# =============================================================================
# NETWORK ANALYSIS
# =============================================================================

def run_network_analyzer(network_suid):
    """Run NetworkAnalyzer to get degree and centrality metrics."""
    print("  Running NetworkAnalyzer...")
    try:
        p4c.analyze_network(directed=False, network=network_suid)
        time.sleep(2)
    except Exception as e:
        print(f"  WARNING: NetworkAnalyzer failed: {e}")

def run_cytohubba(network_suid, method="Degree"):
    """
    Run CytoHubba via Cytoscape Commands API.
    Returns ranked node list from CytoHubba.
    """
    print(f"  Running CytoHubba ({method})...")
    try:
        command = f"cybubba rank network=SUID:{network_suid} method={method} topNodes={TOP_N}"
        result = p4c.commands_post(command)
        time.sleep(2)
        return result
    except Exception as e:
        print(f"  WARNING: CytoHubba {method} failed: {e}")
        return None

def get_hub_genes_from_table(network_suid, method="Degree"):
    """
    Fallback: Get top hub genes directly from node table by sorting Degree column.
    Used when CytoHubba command API is unavailable.
    """
    node_table = p4c.get_node_table(network=network_suid)
    if "Degree" in node_table.columns:
        top = node_table.sort_values("Degree", ascending=False).head(TOP_N)
        return top[["name", "Degree"]].reset_index(drop=True)
    else:
        print("  WARNING: Degree column not found. Run NetworkAnalyzer first.")
        return pd.DataFrame()

# =============================================================================
# VISUALIZATION
# =============================================================================

MODULE_COLOR_MAP = {
    "blue":        "#4169E1",
    "turquoise":   "#40E0D0",
    "brown":       "#8B4513",
    "yellow":      "#FFD700",
    "green":       "#228B22",
    "red":         "#DC143C",
    "black":       "#222222",
    "pink":        "#FF69B4",
    "magenta":     "#FF00FF",
    "purple":      "#800080",
    "greenyellow": "#ADFF2F",
    "tan":         "#D2B48C",
    "salmon":      "#FA8072",
    "cyan":        "#00CED1",
    "midnightblue":"#191970",
    "grey":        "#808080",
}

def apply_visual_style(network_suid, nodes_df):
    """Apply color styling based on module column."""
    print("  Applying visual style...")
    try:
        # Create a new visual style
        style_name = f"HubGene_Style_{network_suid}"
        p4c.create_visual_style(style_name)

        # Map module column to node fill color
        if "module" in nodes_df.columns:
            unique_modules = nodes_df["module"].dropna().unique().tolist()
            mapped_colors = [MODULE_COLOR_MAP.get(m.lower(), "#AAAAAA") for m in unique_modules]

            p4c.set_node_color_mapping(
                table_column="module",
                table_column_values=unique_modules,
                colors=mapped_colors,
                mapping_type="d",
                style_name=style_name,
                network=network_suid
            )

        # Node size by degree
        p4c.set_node_size_mapping(
            table_column="Degree",
            mapping_type="c",
            style_name=style_name,
            network=network_suid
        )

        # Node label = gene name
        p4c.set_node_label_mapping(
            table_column="name",
            style_name=style_name,
            network=network_suid
        )

        p4c.set_visual_style(style_name, network=network_suid)
        p4c.layout_network("force-directed", network=network_suid)
        time.sleep(3)

    except Exception as e:
        print(f"  WARNING: Visual styling failed: {e}")

def export_network_image(network_suid, dataset_name):
    """Export network as PNG image."""
    print("  Exporting network image...")
    output_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_network.png")
    try:
        p4c.export_image(
            filename=output_path,
            type="PNG",
            resolution=150,
            network=network_suid
        )
        print(f"  Image saved: {output_path}")
        return output_path
    except Exception as e:
        print(f"  WARNING: Image export failed: {e}")
        return None

# =============================================================================
# TRANSCRIPTION FACTOR ANALYSIS via ChEA3
# =============================================================================

def get_transcription_factors(hub_genes, dataset_name):
    """Query ChEA3 API with hub gene list to get transcription factors."""
    print(f"  Querying ChEA3 for transcription factors...")
    gene_list = hub_genes["name"].tolist() if "name" in hub_genes.columns else hub_genes.tolist()

    payload = {
        "query_name": dataset_name,
        "gene_set": gene_list
    }
    try:
        response = requests.post(CHEA3_URL, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        # Use MeanRank library (most robust ChEA3 result)
        if "MeanRank" in data:
            tf_df = pd.DataFrame(data["MeanRank"])
            tf_df = tf_df[["TF", "Score", "Overlapping_Genes"]].head(20)
            tf_df.columns = ["Transcription Factor", "Score", "Overlapping Genes"]
            print(f"  Found {len(tf_df)} transcription factors.")
            return tf_df
        else:
            print("  WARNING: Unexpected ChEA3 response format.")
            return pd.DataFrame()
    except Exception as e:
        print(f"  WARNING: ChEA3 query failed: {e}")
        return pd.DataFrame()

# =============================================================================
# EXCEL REPORT GENERATION
# =============================================================================

def build_excel_report(all_results):
    """Build a formatted Excel summary report across all datasets."""
    print("\nGenerating Excel summary report...")
    wb = Workbook()

    # --- Styles ---
    header_font = Font(bold=True, color="FFFFFF", size=11)
    header_fill = PatternFill("solid", start_color="1F4E79")
    subheader_fill = PatternFill("solid", start_color="2E75B6")
    alt_fill = PatternFill("solid", start_color="D6E4F0")
    center = Alignment(horizontal="center", vertical="center")
    thin = Side(style="thin", color="AAAAAA")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def style_header(cell, sub=False):
        cell.font = header_font
        cell.fill = subheader_fill if sub else header_fill
        cell.alignment = center
        cell.border = border

    def style_cell(cell, alt=False):
        if alt:
            cell.fill = alt_fill
        cell.alignment = Alignment(vertical="center")
        cell.border = border

    # --- Summary Sheet ---
    summary_ws = wb.active
    summary_ws.title = "Summary"
    summary_ws.append(["Hub Gene Analysis Summary — Atherosclerosis"])
    summary_ws["A1"].font = Font(bold=True, size=14, color="1F4E79")
    summary_ws.append([f"Datasets analysed: {len(all_results)}"])
    summary_ws.append([f"Top N hub genes per dataset: {TOP_N}"])
    summary_ws.append([f"CytoHubba methods: {', '.join(CYTOHUBBA_METHODS)}"])
    summary_ws.append([])

    # Summary table headers
    summary_ws.append(["Dataset", "Top Hub Gene", "Degree", "Shared with Other Datasets"])
    for col in range(1, 5):
        style_header(summary_ws.cell(6, col))
    summary_ws.column_dimensions["A"].width = 35
    summary_ws.column_dimensions["B"].width = 20
    summary_ws.column_dimensions["C"].width = 12
    summary_ws.column_dimensions["D"].width = 35

    # Collect all hub genes for overlap detection
    all_hub_sets = {r["name"]: set(r["hub_genes"]["name"].tolist())
                    for r in all_results if not r["hub_genes"].empty}

    row = 7
    for i, result in enumerate(all_results):
        if result["hub_genes"].empty:
            continue
        top_gene = result["hub_genes"].iloc[0]["name"]
        top_degree = result["hub_genes"].iloc[0].get("Degree", "N/A")
        my_set = all_hub_sets.get(result["name"], set())
        shared = []
        for other_name, other_set in all_hub_sets.items():
            if other_name != result["name"]:
                overlap = my_set & other_set
                if overlap:
                    shared.append(f"{other_name}: {', '.join(overlap)}")
        shared_str = " | ".join(shared) if shared else "None"

        summary_ws.cell(row, 1, result["name"])
        summary_ws.cell(row, 2, top_gene)
        summary_ws.cell(row, 3, top_degree)
        summary_ws.cell(row, 4, shared_str)
        for col in range(1, 5):
            style_cell(summary_ws.cell(row, col), alt=(i % 2 == 1))
        row += 1

    # --- Per-dataset sheets ---
    for result in all_results:
        name = result["name"][:28]  # Excel sheet name max 31 chars
        ws = wb.create_sheet(title=name)
        ws.column_dimensions["A"].width = 20
        ws.column_dimensions["B"].width = 14
        ws.column_dimensions["C"].width = 14
        ws.column_dimensions["D"].width = 30
        ws.column_dimensions["E"].width = 14

        # Title
        ws.merge_cells("A1:E1")
        ws["A1"] = f"Hub Gene Report: {result['name']}"
        ws["A1"].font = Font(bold=True, size=13, color="1F4E79")
        ws["A1"].alignment = center

        # Hub genes table
        ws["A3"] = "TOP HUB GENES"
        ws["A3"].font = Font(bold=True, size=11, color="2E75B6")

        hub_headers = ["Rank", "Gene", "Degree", "Module", "MCC Score"]
        for col, h in enumerate(hub_headers, 1):
            style_header(ws.cell(4, col), sub=True)
            ws.cell(4, col, h)

        hub_df = result["hub_genes"]
        nodes_df = result["nodes_df"]
        module_map = dict(zip(nodes_df["id"], nodes_df["module"])) if "module" in nodes_df.columns else {}

        for i, (_, row_data) in enumerate(hub_df.iterrows()):
            gene = row_data.get("name", "")
            degree = row_data.get("Degree", "N/A")
            module = module_map.get(gene, "N/A")
            mcc = row_data.get("MCC", "N/A")
            ws.cell(5 + i, 1, i + 1)
            ws.cell(5 + i, 2, gene)
            ws.cell(5 + i, 3, degree)
            ws.cell(5 + i, 4, module)
            ws.cell(5 + i, 5, mcc)
            for col in range(1, 6):
                style_cell(ws.cell(5 + i, col), alt=(i % 2 == 1))

        # Transcription factors table
        tf_start_row = 5 + TOP_N + 2
        ws.cell(tf_start_row, 1, "TOP TRANSCRIPTION FACTORS (ChEA3)")
        ws.cell(tf_start_row, 1).font = Font(bold=True, size=11, color="2E75B6")

        tf_headers = ["Rank", "Transcription Factor", "Score", "Overlapping Genes"]
        for col, h in enumerate(tf_headers, 1):
            style_header(ws.cell(tf_start_row + 1, col), sub=True)
            ws.cell(tf_start_row + 1, col, h)

        tf_df = result.get("tf_df", pd.DataFrame())
        if not tf_df.empty:
            for i, (_, tf_row) in enumerate(tf_df.iterrows()):
                ws.cell(tf_start_row + 2 + i, 1, i + 1)
                ws.cell(tf_start_row + 2 + i, 2, tf_row.get("Transcription Factor", ""))
                ws.cell(tf_start_row + 2 + i, 3, tf_row.get("Score", ""))
                ws.cell(tf_start_row + 2 + i, 4, str(tf_row.get("Overlapping Genes", "")))
                for col in range(1, 5):
                    style_cell(ws.cell(tf_start_row + 2 + i, col), alt=(i % 2 == 1))
        else:
            ws.cell(tf_start_row + 2, 1, "ChEA3 results unavailable — paste hub genes manually at maayanlab.cloud/chea3")

    output_path = os.path.join(OUTPUT_DIR, "hub_gene_summary_report.xlsx")
    wb.save(output_path)
    print(f"Excel report saved: {output_path}")
    return output_path

# =============================================================================
# MAIN PIPELINE
# =============================================================================

def run_pipeline():
    if not check_cytoscape():
        return
    cytohubba_available = check_cytohubba()

    all_results = []

    for dataset in DATASETS:
        result = {
            "name": dataset["name"],
            "hub_genes": pd.DataFrame(),
            "nodes_df": pd.DataFrame(),
            "tf_df": pd.DataFrame(),
        }

        try:
            # 1. Load network
            network_suid, nodes_df, edges_df = load_network(dataset)
            result["nodes_df"] = nodes_df

            # 2. Analyze network (adds Degree column)
            run_network_analyzer(network_suid)

            # 3. Get hub genes
            if cytohubba_available:
                for method in CYTOHUBBA_METHODS:
                    run_cytohubba(network_suid, method=method)
                # After CytoHubba runs, get ranked table from node table
                hub_genes = get_hub_genes_from_table(network_suid, method="Degree")
            else:
                # Fallback to raw degree from node table
                hub_genes = get_hub_genes_from_table(network_suid)

            result["hub_genes"] = hub_genes

            print(f"\n  Top {TOP_N} Hub Genes ({dataset['name']}):")
            print(hub_genes.to_string(index=False))

            # 4. Apply visual style and export image
            apply_visual_style(network_suid, nodes_df)
            export_network_image(network_suid, dataset["name"])

            # 5. Get transcription factors via ChEA3
            if not hub_genes.empty:
                tf_df = get_transcription_factors(hub_genes, dataset["name"])
                result["tf_df"] = tf_df

        except Exception as e:
            print(f"  ERROR processing {dataset['name']}: {e}")

        all_results.append(result)

    # 6. Build Excel summary report
    build_excel_report(all_results)

    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print(f"All outputs saved to: {OUTPUT_DIR}/")
    print("="*60)

if __name__ == "__main__":
    run_pipeline()